# Run the 24-Hour Tromsø GMF Pipeline

Runs the pinned Hardtarget generalized matched-filter workflow on the prepared hourly archives, validates each hourly product, and publishes the campaign outputs.


In [ ]:
from pathlib import Path

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent


## 1. Install the compatible NumPy version


In [ ]:
%pip install --no-cache-dir --force-reinstall "numpy==1.26.4"


## 2. Set repository paths, detect cores, and configure production


In [ ]:
from concurrent.futures import ThreadPoolExecutor, as_completed
from datetime import datetime, timezone
from pathlib import Path
import configparser
import importlib
import json
import os
import re
import shutil
import subprocess
import sys
import tarfile
import threading
import time
import traceback
import uuid

import h5py
import numpy as np
import pandas as pd


                                                                      
MAX_PARALLEL_HOURS = 7
REQUIRED_LOGICAL_CPU_THREADS = 8
PREPROCESS_PARALLEL_LIMIT = 7
MIN_START_FREE_DISK_GIB = 180.0
HEARTBEAT_SECONDS = 60
CLEAN_LOCAL_AFTER_SUCCESS = True
PRODUCTION_PROFILE = "EIGHT_LOGICAL_THREADS_ALL_STAGES_MAX_VERBOSE"

BASE = PROJECT_ROOT
TROMSO_DATA_ROOT = BASE / "data" / "tromso"
ARCHIVE_DIR = TROMSO_DATA_ROOT / "hourly_archives"
ARCHIVE_MANIFEST_DIR = TROMSO_DATA_ROOT / "archive_manifests"
PREPARATION_COMPLETE = ARCHIVE_MANIFEST_DIR / "PREPARATION_COMPLETE.json"

GMF_ROOT = BASE / "processed" / "tromso_2018" / "gmf"
LEGACY_ROOT = BASE / "processed" / "tromso_2018" / "legacy_development" / "20180104_12"
CONTROL_DIR = BASE / "phase0_outputs" / "gmf_parallel_control"

WORK_ROOT = Path(".cache/debris_gmf_direct")
HOUR_WORK_ROOT = WORK_ROOT / "hours"

for directory in [GMF_ROOT, CONTROL_DIR, WORK_ROOT, HOUR_WORK_ROOT]:
    directory.mkdir(parents=True, exist_ok=True)

if hasattr(os, "sched_getaffinity"):
    ALL_VISIBLE_CPU_THREADS = sorted(os.sched_getaffinity(0))
else:
    ALL_VISIBLE_CPU_THREADS = list(range(os.cpu_count() or 1))

if len(ALL_VISIBLE_CPU_THREADS) < REQUIRED_LOGICAL_CPU_THREADS:
    raise RuntimeError(
        f"This notebook requires the CPU + High-RAM runtime with at least "
        f"{REQUIRED_LOGICAL_CPU_THREADS} logical CPU threads; detected "
        f"{len(ALL_VISIBLE_CPU_THREADS)}: {ALL_VISIBLE_CPU_THREADS}"
    )

VISIBLE_CORES = ALL_VISIBLE_CPU_THREADS[:MAX_PARALLEL_HOURS]
LOCAL_FREE_DISK_GIB = shutil.disk_usage('.').free / 2**30
if LOCAL_FREE_DISK_GIB < MIN_START_FREE_DISK_GIB:
    raise RuntimeError(
        f"Only {LOCAL_FREE_DISK_GIB:.1f} GiB is free under .; "
        f"eight-hour production requires at least {MIN_START_FREE_DISK_GIB:.1f} GiB."
    )

print(f"Detected logical CPU threads: {ALL_VISIBLE_CPU_THREADS}")
print(f"Production CPU assignments: {VISIBLE_CORES}")
print(f"This run will start up to {MAX_PARALLEL_HOURS} pending hours concurrently.")
print(f"Preprocessing concurrency: {PREPROCESS_PARALLEL_LIMIT} hours (all workers)")
print(f"Local free disk: {LOCAL_FREE_DISK_GIB:.1f} GiB")
print("Final output root:", GMF_ROOT)


## 3. Install the pinned Hardtarget revision


In [ ]:
HARDTARGET_REPOSITORY = "https://github.com/danielk333/hardtarget.git"
HARDTARGET_COMMIT = "ffdf6cee99f021d554c34dec50360cf614864896"
HARDTARGET_SOURCE = Path(".cache/hardtarget_pinned")


def run_live(command, *, cwd=None, env=None, check=True):
    command = [str(part) for part in command]
    print("$", " ".join(command))
    completed = subprocess.run(command, cwd=cwd, env=env)
    if check and completed.returncode != 0:
        raise RuntimeError(
            f"Command failed with return code {completed.returncode}: {' '.join(command)}"
        )
    return completed


run_live(["apt-get", "update", "-qq"])
run_live([
    "apt-get", "install", "-y", "-qq",
    "gcc", "g++", "git", "bzip2", "rsync", "util-linux",
    "libfftw3-dev", "libhdf5-dev", "openmpi-bin", "libopenmpi-dev",
])

if HARDTARGET_SOURCE.exists():
    shutil.rmtree(HARDTARGET_SOURCE)

run_live(["git", "clone", "--filter=blob:none", HARDTARGET_REPOSITORY, HARDTARGET_SOURCE])
run_live(["git", "checkout", "--detach", HARDTARGET_COMMIT], cwd=HARDTARGET_SOURCE)

actual_commit = subprocess.check_output(
    ["git", "rev-parse", "HEAD"], cwd=HARDTARGET_SOURCE, text=True
).strip()
assert actual_commit == HARDTARGET_COMMIT, (actual_commit, HARDTARGET_COMMIT)

utils_path = HARDTARGET_SOURCE / "src" / "hardtarget" / "utils.py"
utils_text = utils_path.read_text(encoding="utf-8")
replacement = "dt.datetime.fromtimestamp(ts, tz=dt.timezone.utc)"
old_variants = [
    "dt.datetime.fromtimestamp(ts, tzinfo=dt.utc)",
    "dt.datetime.fromtimestamp(ts, tz=dt.utc)",
]

for old in old_variants:
    if old in utils_text:
        utils_text = utils_text.replace(old, replacement)

if replacement not in utils_text:
    raise RuntimeError("The expected Hardtarget timezone line was not found.")

utils_path.write_text(utils_text, encoding="utf-8")
HARDTARGET_PATCH = subprocess.check_output(
    ["git", "diff", "--", "src/hardtarget/utils.py"],
    cwd=HARDTARGET_SOURCE,
    text=True,
)

run_live([sys.executable, "-m", "pip", "install", "-q", "--upgrade", "pip"])
run_live([sys.executable, "-m", "pip", "install", "-q", "mpi4py"])
run_live([sys.executable, "-m", "pip", "install", "-q", ".[plotting]"], cwd=HARDTARGET_SOURCE)

import hardtarget
import hardtarget.utils as ht_utils

importlib.invalidate_caches()
ht_utils = importlib.reload(ht_utils)

print("Hardtarget module:", hardtarget.__file__)
print("Pinned commit:", actual_commit)
print("Timezone smoke test:", ht_utils.str_from_ts(1515070800.0))
run_live(["hardtarget", "--version"])
run_live(["hardtarget", "analyze", "-h"])


## 4. Read the existing verified archive inventory (no checksum recomputation)


In [ ]:
GMF_CONFIG_TEXT = '[signal-processing]\nn_ipp = 10\nipp_offset = 0\nmin_range_gate = 6800\nmax_range_gate = 7280\nrange_gate_step = 1\nfrequency_decimation = 16\nclutter_length = 0\nmin_acceleration = -300.0\nmax_acceleration = 300.0\nnum_cohints_per_file = 100\nnode_gpus = 0\ndpt_ipp_delay_parameter = 5\n'
N_IPP = 10
IPP_SAMPLES = 20000
SAMPLE_RATE_HZ = 1000000
COHINT_SECONDS = N_IPP * IPP_SAMPLES / SAMPLE_RATE_HZ
assert abs(COHINT_SECONDS - 0.2) < 1e-12

def utc_now():
    return datetime.now(timezone.utc).isoformat()

def parse_iso_utc(value):
    parsed = datetime.fromisoformat(str(value).replace('Z', '+00:00'))
    if parsed.tzinfo is None:
        parsed = parsed.replace(tzinfo=timezone.utc)
    return parsed.astimezone(timezone.utc)

def format_iso_for_hardtarget(value):
    value = value.astimezone(timezone.utc).replace(tzinfo=None)
    return value.isoformat(timespec='microseconds')

def is_within(path, root):
    path = Path(path).resolve()
    root = Path(root).resolve()
    return path == root or root in path.parents

def remove_local_tree(path):
    path = Path(path)
    if not is_within(path, WORK_ROOT):
        raise RuntimeError(f'Refusing to delete outside local work root: {path}')
    if path.exists():
        shutil.rmtree(path)

def remove_local_file(path):
    path = Path(path)
    if not is_within(path, WORK_ROOT):
        raise RuntimeError(f'Refusing to delete outside local work root: {path}')
    if path.exists():
        path.unlink()

def tail_text(path, max_chars=8000):
    path = Path(path)
    if not path.exists():
        return ''
    text = path.read_text(encoding='utf-8', errors='replace')
    return text[-max_chars:]

def thread_limited_env(extra=None):
    env = os.environ.copy()
    env.update({'OMPI_ALLOW_RUN_AS_ROOT': '1', 'OMPI_ALLOW_RUN_AS_ROOT_CONFIRM': '1', 'OMP_NUM_THREADS': '1', 'OPENBLAS_NUM_THREADS': '1', 'MKL_NUM_THREADS': '1', 'NUMEXPR_NUM_THREADS': '1', 'PYTHONUNBUFFERED': '1', 'PYTHONIOENCODING': 'utf-8'})
    if extra:
        env.update(extra)
    return env

def run_logged(command, log_path, label, *, env=None, heartbeat_seconds=HEARTBEAT_SECONDS):
    command = [str(part) for part in command]
    log_path = Path(log_path)
    log_path.parent.mkdir(parents=True, exist_ok=True)
    started = time.monotonic()
    with open(log_path, 'w', encoding='utf-8') as handle:
        handle.write(f"[{utc_now()}] $ {' '.join(command)}\n")
        handle.flush()
        process = subprocess.Popen(command, stdout=handle, stderr=subprocess.STDOUT, text=True, env=env)
        while True:
            try:
                return_code = process.wait(timeout=heartbeat_seconds)
                break
            except subprocess.TimeoutExpired:
                elapsed = (time.monotonic() - started) / 60
                print(f'[{label}] still running — {elapsed:.1f} min elapsed')
        elapsed_seconds = time.monotonic() - started
        handle.write(f'\n[{utc_now()}] RETURN CODE: {return_code}\n')
        handle.write(f'ELAPSED_SECONDS: {elapsed_seconds:.3f}\n')
    if return_code != 0:
        print(tail_text(log_path))
        raise RuntimeError(f"{label} failed with return code {return_code}: {' '.join(command)}")
    print(f'[{label}] finished in {elapsed_seconds / 60:.1f} min')
    return elapsed_seconds
if not PREPARATION_COMPLETE.is_file():
    raise FileNotFoundError("Missing PREPARATION_COMPLETE.json. Run 'research Prepare Tromso 24h Archives' first.")
preparation = json.loads(PREPARATION_COMPLETE.read_text(encoding='utf-8'))
expected_hours = list(preparation.get('expected_hours', []))
if preparation.get('status') != 'complete' or len(expected_hours) != 24:
    raise RuntimeError(f'Invalid preparation marker: {PREPARATION_COMPLETE}')
inventory_rows = []
for hour_key in expected_hours:
    manifest_path = ARCHIVE_MANIFEST_DIR / f'{hour_key}.json'
    manifest = json.loads(manifest_path.read_text(encoding='utf-8'))
    if manifest.get('status') != 'verified' or manifest.get('hour_key') != hour_key:
        raise RuntimeError(f'Manifest is not verified for {hour_key}')
    archive_path = ARCHIVE_DIR / manifest['archive_name']
    if not archive_path.is_file():
        raise FileNotFoundError(archive_path)
    if archive_path.stat().st_size != int(manifest['archive_size_bytes']):
        raise RuntimeError(f'Archive size changed for {hour_key}: {archive_path}')
    inventory_rows.append({'hour_key': hour_key, 'archive_name': archive_path.name, 'archive_path': str(archive_path), 'archive_gib': archive_path.stat().st_size / 2 ** 30, 'member_files': int(manifest['member_files'])})
archive_inventory = pd.DataFrame(inventory_rows).sort_values('hour_key').reset_index(drop=True)
print('FAST PREFLIGHT PASSED:', len(archive_inventory), 'prepared hourly archives')
display(archive_inventory[['hour_key', 'archive_name', 'archive_gib', 'member_files']])


## 5. Select the next pending hour for every available core


In [ ]:
LEGACY_COMPLETE_HOURS = {"20180104_12"}


def completed_hours_from_outputs():
    completed = set(LEGACY_COMPLETE_HOURS)
    if GMF_ROOT.exists():
        for child in GMF_ROOT.iterdir():
            if child.is_dir() and (child / "COMPLETE.json").is_file():
                completed.add(child.name)
    return completed


def hardtarget_analyze_command(drf_root, config_path, output_dir, start_dt, end_dt, core_id):
    return [
        "taskset", "-c", str(core_id),
        "mpirun", "--allow-run-as-root", "--bind-to", "none", "--oversubscribe",
        "-np", "1",
        "hardtarget", "-v", "analyze",
        str(drf_root), "uhf",
        "--config", str(config_path),
        "-o", str(output_dir),
        "-p",
        "-s", format_iso_for_hardtarget(start_dt),
        "-e", format_iso_for_hardtarget(end_dt),
        "-m", "fgmf",
        "-i", "c",
        "--clobber",
    ]


completed_before_run = completed_hours_from_outputs()
all_hours = archive_inventory["hour_key"].tolist()
pending_hours = [hour for hour in all_hours if hour not in completed_before_run]
worker_count = min(MAX_PARALLEL_HOURS, len(VISIBLE_CORES), len(pending_hours))
selected_hours = pending_hours[:worker_count]

selected_rows = [
    {"hour_key": hour_key, "core_ids": [core_id], "mpi_ranks": 1}
    for hour_key, core_id in zip(selected_hours, VISIBLE_CORES)
]
resource_plan = {
    "profile": PRODUCTION_PROFILE,
    "detected_logical_cpu_threads": len(ALL_VISIBLE_CPU_THREADS),
    "assigned_logical_cpu_threads": list(VISIBLE_CORES),
    "preprocess_parallel_limit": PREPROCESS_PARALLEL_LIMIT,
    "selected": selected_rows,
}

print("Already complete/protected:", sorted(completed_before_run))
print("Pending before this run:", pending_hours)
if selected_rows:
    print(f"STARTING NOW: {len(selected_rows)} hour(s), one per logical CPU thread")
    display(pd.DataFrame(selected_rows))
else:
    print("No pending GMF hours remain.")


## 6. Per-hour conversion, GMF, validation, and publishing


In [ ]:
POSTPROCESS_LOCK = threading.Lock()
PREPROCESS_SEMAPHORE = threading.Semaphore(PREPROCESS_PARALLEL_LIMIT)


def safe_tar_members(tar, extract_root):
    extract_root = Path(extract_root).resolve()
    members = []
    for member in tar.getmembers():
        destination = (extract_root / member.name).resolve()
        if destination != extract_root and extract_root not in destination.parents:
            raise RuntimeError(f"Unsafe archive member path: {member.name}")
        members.append(member)
    return members


def stage_archive(archive_path, hour_work, label):
    archive_path = Path(archive_path)
    local_archive = hour_work / "archive" / archive_path.name
    local_archive.parent.mkdir(parents=True, exist_ok=True)

    if local_archive.exists() and local_archive.stat().st_size != archive_path.stat().st_size:
        remove_local_file(local_archive)

    if not local_archive.exists():
        run_logged(
            ["rsync", "-a", "--info=progress2", str(archive_path), str(local_archive)],
            hour_work / "result" / "logs" / "archive_stage.log",
            f"{label} archive stage",
            env=thread_limited_env(),
        )

    if local_archive.stat().st_size != archive_path.stat().st_size:
        raise RuntimeError(f"Local archive size mismatch for {label}")
    return local_archive


def extract_hour(local_archive, hour_key, hour_work):
    extract_root = hour_work / "extracted"
    extract_root.mkdir(parents=True, exist_ok=True)

    with tarfile.open(local_archive, "r:") as tar:
        members = safe_tar_members(tar, extract_root)
        for member_number, member in enumerate(members, start=1):
            try:
                tar.extract(member, extract_root, filter="data")
            except TypeError:
                tar.extract(member, extract_root)
            print_status(
                hour_key,
                "archive extraction",
                f"member {member_number}/{len(members)}: {member.name}",
            )

    matching_dirs = [path for path in extract_root.rglob(hour_key) if path.is_dir()]
    if len(matching_dirs) != 1:
        raise RuntimeError(f"Expected one extracted {hour_key} directory, found {matching_dirs}")

    hour_dir = matching_dirs[0]
    mat_files = sorted(list(hour_dir.glob("*.mat")) + list(hour_dir.glob("*.mat.bz2")))
    if len(mat_files) < 240:
        raise RuntimeError(f"Only {len(mat_files)} MAT files found for {hour_key}")
    return hour_dir.parent, len(mat_files)


def extract_gmf_peak_table(gmf_dir):
    gmf_files = sorted(Path(gmf_dir).rglob("*.h5"))
    rows = []

    hour_key = Path(gmf_dir).parents[1].name
    for file_number, file_path in enumerate(gmf_files, start=1):
        print_status(
            hour_key,
            "GMF postprocessing",
            f"reading HDF5 {file_number}/{len(gmf_files)}: {file_path.name}",
        )
        with h5py.File(file_path, "r") as handle:
            required = [
                "gmf_peak", "range_peak", "range_rate_peak",
                "acceleration_peak", "integration_index", "tx_power",
            ]
            missing = [name for name in required if name not in handle]
            if missing:
                raise RuntimeError(f"{file_path.name} is missing datasets: {missing}")

            arrays = {name: np.asarray(handle[name][()]) for name in required}
            n_rows = len(arrays["gmf_peak"])
            if any(len(value) != n_rows for value in arrays.values()):
                raise RuntimeError(f"Dataset length mismatch in {file_path.name}")

            match = re.search(r"gmf-(\d+)\.h5$", file_path.name)
            if not match:
                raise RuntimeError(f"Cannot parse GMF timestamp: {file_path.name}")
            file_start_unix = int(match.group(1)) / 1e6
            pointing = np.asarray(handle["pointing"][()]) if "pointing" in handle else None

            for index in range(n_rows):
                integration_index = int(arrays["integration_index"][index])
                row = {
                    "source_file": file_path.name,
                    "file_start_unix": file_start_unix,
                    "integration_index": integration_index,
                    "time_unix": file_start_unix + integration_index * COHINT_SECONDS,
                    "gmf_peak": float(arrays["gmf_peak"][index]),
                    "range_peak_m": float(arrays["range_peak"][index]),
                    "range_rate_peak_mps": float(arrays["range_rate_peak"][index]),
                    "acceleration_peak_mps2": float(arrays["acceleration_peak"][index]),
                    "tx_power": float(arrays["tx_power"][index]),
                }
                if pointing is not None:
                    row["pointing_0"] = float(pointing[index, 0])
                    row["pointing_1"] = float(pointing[index, 1])
                rows.append(row)

    table = pd.DataFrame(rows).sort_values("time_unix").reset_index(drop=True)
    return gmf_files, table


def publish_result(result_dir, hour_key, expected_h5_files, expected_h5_bytes):
    final_dir = GMF_ROOT / hour_key
    if final_dir.exists():
        raise RuntimeError(f"Refusing to replace existing existing output: {final_dir}")

    staging_dir = GMF_ROOT / f"_staging_{hour_key}_{uuid.uuid4().hex[:10]}"
    staging_dir.mkdir(parents=True, exist_ok=False)

    try:
        run_logged(
            ["rsync", "-a", "--info=progress2", f"{result_dir}/", f"{staging_dir}/"],
            CONTROL_DIR / f"sync_{hour_key}_{datetime.now(timezone.utc):%Y%m%dT%H%M%SZ}.log",
            f"{hour_key} output publish",
            env=thread_limited_env(),
        )

        staged_h5 = sorted((staging_dir / "gmf").rglob("*.h5"))
        staged_bytes = sum(path.stat().st_size for path in staged_h5)
        if len(staged_h5) != expected_h5_files or staged_bytes != expected_h5_bytes:
            raise RuntimeError(
                f"output staging mismatch for {hour_key}: "
                f"files {len(staged_h5)}/{expected_h5_files}, bytes {staged_bytes}/{expected_h5_bytes}"
            )
        if not (staging_dir / "COMPLETE.json").is_file():
            raise RuntimeError("output staging folder lacks COMPLETE.json")

        staging_dir.rename(final_dir)
    except Exception:
        print("Staging folder retained for inspection:", staging_dir)
        raise

    return final_dir


def process_hour(hour_key, archive_path, core_ids):
    started_at = utc_now()
    transaction_started = time.monotonic()
    hour_work = HOUR_WORK_ROOT / hour_key
    remove_local_tree(hour_work)

    result_dir = hour_work / "result"
    log_dir = result_dir / "logs"
    gmf_dir = result_dir / "gmf"
    log_dir.mkdir(parents=True, exist_ok=True)
    gmf_dir.mkdir(parents=True, exist_ok=True)

    timings = {}
    cpu_list = ",".join(str(core) for core in core_ids)
    ranks = len(core_ids)
    label = f"{hour_key} / CPUs {cpu_list}"
    preprocess_slot_acquired = False

    try:
        print_status(
            hour_key,
            "preprocessing",
            f"requesting one of {PREPROCESS_PARALLEL_LIMIT} slots",
        )
        PREPROCESS_SEMAPHORE.acquire()
        preprocess_slot_acquired = True
        print_status(
            hour_key,
            "preprocessing",
            f"slot acquired; free disk={shutil.disk_usage('.').free / 2**30:.1f} GiB",
        )

        t0 = time.monotonic()
        local_archive = stage_archive(archive_path, hour_work, label)
        timings["archive_stage_seconds"] = time.monotonic() - t0

        t0 = time.monotonic()
        source_root, mat_file_count = extract_hour(local_archive, hour_key, hour_work)
        timings["extract_seconds"] = time.monotonic() - t0

        run_name = f"eiscat_uhf_{hour_key}"
        drf_parent = hour_work / "drf"
        drf_parent.mkdir(parents=True, exist_ok=True)
        drf_dir = drf_parent / run_name

        t0 = time.monotonic()
        run_logged(
            [
                "taskset", "-c", str(core_ids[0]),
                "hardtarget", "-v", "convert", "eiscat",
                str(source_root), str(drf_parent), "-p", "-n", run_name,
            ],
            log_dir / "hardtarget_convert.log",
            f"{label} MAT→DRF",
            env=thread_limited_env(),
        )
        timings["convert_seconds"] = time.monotonic() - t0

        metadata_path = drf_dir / "metadata.ini"
        rf_files = sorted(drf_dir.rglob("rf@*.h5"))
        if not metadata_path.is_file() or not rf_files:
            raise RuntimeError(f"Digital RF validation failed for {hour_key}")

        run_logged(
            ["hardtarget", "-v", "inspect", "drf", str(drf_dir)],
            log_dir / "hardtarget_inspect.log",
            f"{label} DRF inspect",
            env=thread_limited_env(),
        )

        metadata = configparser.ConfigParser()
        metadata.read(metadata_path)
        start_time = metadata["Bounds"]["start"]
        end_time = metadata["Bounds"]["end"]

        config_path = result_dir / "gmf_config.ini"
        config_path.write_text(GMF_CONFIG_TEXT, encoding="utf-8")
        (result_dir / "hardtarget_patch.diff").write_text(HARDTARGET_PATCH, encoding="utf-8")
        shutil.copy2(metadata_path, result_dir / "drf_metadata.ini")

                                                                        
        remove_local_file(local_archive)
        remove_local_tree(hour_work / "archive")
        remove_local_tree(hour_work / "extracted")
        print_status(
            hour_key,
            "preprocessing",
            "removed local TAR and extracted MAT files before GMF",
        )
        PREPROCESS_SEMAPHORE.release()
        preprocess_slot_acquired = False
        print_status(
            hour_key,
            "preprocessing",
            f"slot released; entering GMF; free disk={shutil.disk_usage('.').free / 2**30:.1f} GiB",
        )

        t0 = time.monotonic()
        run_logged(
            hardtarget_analyze_command(
                drf_dir,
                config_path,
                gmf_dir,
                parse_iso_utc(start_time),
                parse_iso_utc(end_time),
                core_id=core_ids[0],
            ),
            log_dir / "hardtarget_gmf.log",
            f"{label} GMF",
            env=thread_limited_env(),
        )
        timings["gmf_seconds"] = time.monotonic() - t0

        t0 = time.monotonic()
        with POSTPROCESS_LOCK:
            gmf_files, peaks = extract_gmf_peak_table(gmf_dir)
        timings["postprocess_seconds"] = time.monotonic() - t0

        if not gmf_files or peaks.empty:
            raise RuntimeError(f"GMF output is empty for {hour_key}")

        start_dt = parse_iso_utc(start_time)
        end_dt = parse_iso_utc(end_time)
        expected_rows = int(round((end_dt - start_dt).total_seconds() / COHINT_SECONDS))
        row_difference = len(peaks) - expected_rows
        if abs(row_difference) > 2:
            raise RuntimeError(
                f"GMF row count {len(peaks)} differs from expected {expected_rows} by {row_difference}"
            )

        numeric_columns = [
            "time_unix", "gmf_peak", "range_peak_m", "range_rate_peak_mps",
            "acceleration_peak_mps2", "tx_power",
        ]
        if not np.isfinite(peaks[numeric_columns].to_numpy()).all():
            raise RuntimeError(f"Non-finite GMF peak-table values for {hour_key}")

        peaks.to_csv(result_dir / "gmf_peaks.csv.gz", index=False, compression="gzip")
        h5_bytes = sum(path.stat().st_size for path in gmf_files)
        completed_at = utc_now()
        timings["transaction_before_publish_seconds"] = time.monotonic() - transaction_started

        manifest = {
            "status": "complete",
            "hour_key": hour_key,
            "started_at_utc": started_at,
            "completed_at_utc": completed_at,
            "input": {
                "drive_archive": str(archive_path),
                "archive_name": Path(archive_path).name,
                "archive_size_bytes": Path(archive_path).stat().st_size,
                "hour_member_file_count": mat_file_count,
                "archive_checksum_recomputed": False,
                "verification_source": "existing verified archive manifest",
            },
            "software": {
                "hardtarget_repository": HARDTARGET_REPOSITORY,
                "hardtarget_commit": HARDTARGET_COMMIT,
                "python": sys.version,
                "numpy": np.__version__,
                "pandas": pd.__version__,
                "h5py": h5py.__version__,
            },
            "runtime": {
                "production_profile": PRODUCTION_PROFILE,
                "cpu_ids": list(core_ids),
                "mpi_ranks": ranks,
                "timings": timings,
            },
            "drf": {
                "native_metadata_start": start_time,
                "native_metadata_end": end_time,
                "rf_file_count": len(rf_files),
            },
            "gmf": {
                "method": "fgmf",
                "implementation": "c",
                "mpi_ranks": ranks,
                "cohint_seconds": COHINT_SECONDS,
                "hdf5_file_count": len(gmf_files),
                "hdf5_total_bytes": h5_bytes,
                "peak_rows": len(peaks),
                "expected_peak_rows": expected_rows,
                "row_difference": row_difference,
                "thresholding_applied": False,
            },
            "validation": {
                "native_metadata_present": True,
                "hardtarget_inspect_passed": True,
                "all_required_values_finite": True,
                "drive_publish_verified": False,
            },
        }
        (result_dir / "manifest.json").write_text(json.dumps(manifest, indent=2), encoding="utf-8")
        (result_dir / "COMPLETE.json").write_text(
            json.dumps({
                "hour_key": hour_key,
                "status": "complete",
                "peak_rows": len(peaks),
                "gmf_hdf5_files": len(gmf_files),
                "completed_at_utc": completed_at,
                "manifest": "manifest.json",
            }, indent=2),
            encoding="utf-8",
        )

        t0 = time.monotonic()
        final_dir = publish_result(result_dir, hour_key, len(gmf_files), h5_bytes)
        timings["publish_seconds"] = time.monotonic() - t0
        timings["transaction_total_seconds"] = time.monotonic() - transaction_started

        final_manifest_path = final_dir / "manifest.json"
        final_manifest = json.loads(final_manifest_path.read_text(encoding="utf-8"))
        final_manifest["runtime"]["timings"] = timings
        final_manifest["validation"]["drive_publish_verified"] = True
        final_manifest_path.write_text(json.dumps(final_manifest, indent=2), encoding="utf-8")

        if CLEAN_LOCAL_AFTER_SUCCESS:
            remove_local_tree(hour_work)

        print(f"[{label}] COMPLETE — {len(gmf_files)} HDF5 files, {len(peaks)} rows")
        return {
            "hour_key": hour_key,
            "status": "complete",
            "mpi_ranks": ranks,
            "core_ids": list(core_ids),
            "gmf_files": len(gmf_files),
            "peak_rows": len(peaks),
            "total_hours": timings["transaction_total_seconds"] / 3600,
            "drive_result": str(final_dir),
        }

    except Exception as error:
        if preprocess_slot_acquired:
            PREPROCESS_SEMAPHORE.release()
            preprocess_slot_acquired = False
            print(f"[{label}] preprocessing slot released after failure")

        failure_record = {
            "hour_key": hour_key,
            "status": "failed",
            "failed_at_utc": utc_now(),
            "archive_path": str(archive_path),
            "cpu_ids": list(core_ids),
            "error_type": type(error).__name__,
            "error": str(error),
            "traceback": traceback.format_exc(),
            "local_work_retained": str(hour_work),
        }
        failure_path = CONTROL_DIR / (
            f"FAILURE_{hour_key}_{datetime.now(timezone.utc):%Y%m%dT%H%M%SZ}.json"
        )
        failure_path.write_text(json.dumps(failure_record, indent=2), encoding="utf-8")
        print(f"[{label}] FAILED — record: {failure_path}")
        print(f"[{label}] local work retained: {hour_work}")
        raise


In [ ]:
for row in selected_rows:
    hour_key = row["hour_key"]
    stale_path = HOUR_WORK_ROOT / hour_key

    if stale_path.exists():
        shutil.rmtree(stale_path)

    print(f"Cleared interrupted local work for {hour_key}", flush=True)


## 7. Launch the selected hours concurrently


In [ ]:
production_results = []

                                                                                           
                                             
CONSOLE_PRINT_LOCK = threading.Lock()
HOUR_TO_CPU = {
    row["hour_key"]: row["core_ids"][0]
    for row in selected_rows
}


def print_status(hour_key, step, message):
    cpu_id = HOUR_TO_CPU.get(hour_key, "?")
    with CONSOLE_PRINT_LOCK:
        print(
            f"[CPU {cpu_id} | {hour_key} | {step}] {message}",
            flush=True,
        )


def parse_run_label(label):
    match = re.search(r"(20\d{6}_\d{2})", label)
    hour_key = match.group(1) if match else "unknown_hour"

    step = label
    if match:
        step = step.replace(hour_key, "", 1)

    step = re.sub(
        r"^[\s/\-]*CPUs?\s+[0-9,\s]+",
        "",
        step,
    ).strip(" /-")

    return hour_key, step or "command"


                                                                    
                                                                      
def run_logged(
    command,
    log_path,
    label,
    *,
    env=None,
    heartbeat_seconds=None,
):
    command = [str(part) for part in command]
    log_path = Path(log_path)
    log_path.parent.mkdir(parents=True, exist_ok=True)

    hour_key, step = parse_run_label(label)
    started = time.monotonic()

    print_status(hour_key, step, "START")
    print_status(hour_key, step, "$ " + " ".join(command))

    with open(log_path, "w", encoding="utf-8") as handle:
        handle.write(f"[{utc_now()}] $ {' '.join(command)}\n")
        handle.flush()

        process = subprocess.Popen(
            command,
            stdout=subprocess.PIPE,
            stderr=subprocess.STDOUT,
            text=True,
            bufsize=1,
            env=env,
        )

        assert process.stdout is not None

                                                                             
        for line in process.stdout:
            handle.write(line)
            handle.flush()

            line = line.rstrip()
            if line:
                print_status(hour_key, step, line)

        return_code = process.wait()
        elapsed_seconds = time.monotonic() - started

        handle.write(f"\n[{utc_now()}] RETURN CODE: {return_code}\n")
        handle.write(f"ELAPSED_SECONDS: {elapsed_seconds:.3f}\n")
        handle.flush()

    if return_code != 0:
        print_status(
            hour_key,
            step,
            f"FAILED — return code {return_code}",
        )
        raise RuntimeError(
            f"{label} failed with return code {return_code}: "
            + " ".join(command)
        )

    print_status(
        hour_key,
        step,
        f"DONE — {elapsed_seconds / 60:.1f} min",
    )
    return elapsed_seconds


                                                                         
                                               
if "_BASE_EXTRACT_HOUR" not in globals():
    _BASE_EXTRACT_HOUR = extract_hour


def extract_hour(local_archive, hour_key, hour_work):
    print_status(hour_key, "archive extraction", "START")

    try:
        result = _BASE_EXTRACT_HOUR(
            local_archive,
            hour_key,
            hour_work,
        )
    except Exception as error:
        print_status(
            hour_key,
            "archive extraction",
            f"FAILED — {error}",
        )
        raise

    print_status(
        hour_key,
        "archive extraction",
        f"DONE — {result[1]} MAT files",
    )
    return result


                                                                 
if "_BASE_EXTRACT_GMF_PEAK_TABLE" not in globals():
    _BASE_EXTRACT_GMF_PEAK_TABLE = extract_gmf_peak_table


def extract_gmf_peak_table(gmf_dir):
    gmf_dir = Path(gmf_dir)
    hour_key = gmf_dir.parents[1].name

    print_status(hour_key, "GMF postprocessing", "START")

    try:
        gmf_files, peaks = _BASE_EXTRACT_GMF_PEAK_TABLE(gmf_dir)
    except Exception as error:
        print_status(
            hour_key,
            "GMF postprocessing",
            f"FAILED — {error}",
        )
        raise

    print_status(
        hour_key,
        "GMF postprocessing",
        f"DONE — {len(gmf_files)} HDF5 files, {len(peaks)} rows",
    )
    return gmf_files, peaks


def run_independent_worker(row, inventory_by_hour):
    hour_key = row["hour_key"]
    core_ids = list(row["core_ids"])
    archive_path = inventory_by_hour.at[hour_key, "archive_path"]

    print_status(hour_key, "worker", "STARTING")

    result = process_hour(
        hour_key,
        archive_path,
        core_ids,
    )

    print_status(
        hour_key,
        "worker",
        f"COMPLETE — output saved to {result['drive_result']}",
    )
    return result


if not selected_rows:
    print("Nothing to launch.")
else:
    inventory_by_hour = archive_inventory.set_index("hour_key")

    print(
        f"Launching {len(selected_rows)} independently pinned logical-thread workers...",
        flush=True,
    )

    for row in selected_rows:
        print_status(
            row["hour_key"],
            "assignment",
            f"assigned exclusively to logical CPU {row['core_ids'][0]}",
        )

    with ThreadPoolExecutor(
        max_workers=len(selected_rows)
    ) as executor:

        future_to_hour = {
            executor.submit(
                run_independent_worker,
                row,
                inventory_by_hour,
            ): row["hour_key"]
            for row in selected_rows
        }

                                                                    
                                            
        for future in as_completed(future_to_hour):
            hour_key = future_to_hour[future]

            try:
                production_results.append(future.result())
            except Exception as error:
                print_status(
                    hour_key,
                    "worker",
                    f"FAILED — {error}",
                )
                production_results.append({
                    "hour_key": hour_key,
                    "status": "failed",
                    "error": str(error),
                })

    production_results = (
        pd.DataFrame(production_results)
        .sort_values("hour_key")
        .reset_index(drop=True)
    )
    display(production_results)

    if (production_results["status"] != "complete").any():
        raise RuntimeError(
            "At least one hour failed. Successful hours remain complete; "
            "rerun later to retry only pending work."
        )


## 8. Campaign status


In [ ]:
status_rows = []
completed_now = completed_hours_from_outputs()

for hour_key in archive_inventory["hour_key"]:
    final_dir = GMF_ROOT / hour_key
    marker = final_dir / "COMPLETE.json"
    if marker.is_file():
        payload = json.loads(marker.read_text(encoding="utf-8"))
        status_rows.append({
            "hour_key": hour_key,
            "status": "complete",
            "gmf_hdf5_files": payload.get("gmf_hdf5_files"),
            "peak_rows": payload.get("peak_rows"),
            "location": str(final_dir),
        })
    elif hour_key in LEGACY_COMPLETE_HOURS:
        status_rows.append({
            "hour_key": hour_key,
            "status": "legacy development hour",
            "gmf_hdf5_files": 1798,
            "peak_rows": 17980,
            "location": str(LEGACY_ROOT / "gmf_full_hour"),
        })
    else:
        status_rows.append({
            "hour_key": hour_key,
            "status": "pending",
            "gmf_hdf5_files": None,
            "peak_rows": None,
            "location": None,
        })

processing_status = pd.DataFrame(status_rows)
display(processing_status)
print(processing_status["status"].value_counts())

completed_count = int((processing_status["status"] != "pending").sum())
print(f"Campaign GMF progress: {completed_count} / 24 hours")
print("Next pending:", processing_status.loc[processing_status.status == "pending", "hour_key"].head(8).tolist())


## Outputs
